# 🎵 Demucs Custom - Обучение на Google Colab

Этот notebook дообучает модель Demucs на твоих данных.

**Инструкция:** Нажимай кнопку ▶️ (Run) на каждой ячейке по очереди сверху вниз.
Не пропускай ячейки!

In [ ]:
# Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive подключен!")
print("Твои веса будут сохраняться в: MyDrive/demucs_custom/")

In [ ]:
# Клонирование проекта с GitHub
#
# Команды запускаем через subprocess, а не через «!».
# Такой код одинаково работает и в Colab, и в обычном Jupyter/VS Code.
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/IgorCim/demucs_custom.git"
PROJECT_DIR = "/content/demucs_custom"

if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print("✅ Код клонирован. Текущая папка:", os.getcwd())
print("Файлы проекта:", sorted(os.listdir('.')))

In [ ]:
# Установка зависимостей
print("⏳ Ставлю зависимости, это 2-4 минуты...")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "demucs", "torchaudio", "librosa", "soundfile", "musdb"], check=True)

import torch
import torchaudio
import demucs
import librosa
import soundfile

print("✅ Все зависимости установлены!")
print("torch     :", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("demucs    :", demucs.__version__)

In [ ]:
# Проверка GPU
import torch

print("=" * 50)
print("ПРОВЕРКА ОБОРУДОВАНИЯ")
print("=" * 50)
if torch.cuda.is_available():
    print(f"✅ GPU доступен: {torch.cuda.get_device_name(0)}")
    print(f"✅ Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("Можно запускать обучение!")
else:
    print("❌ GPU НЕ доступен!")
    print("Что делать:")
    print("1. Меню Runtime → Change runtime type")
    print("2. Hardware accelerator → GPU T4")
    print("3. Save → перезапусти все ячейки")
    raise RuntimeError("Без GPU обучение не имеет смысла. Сделай шаги выше и перезапусти.")

In [ ]:
# Настройка путей к Google Drive
from pathlib import Path

drive_models_path = '/content/drive/MyDrive/demucs_custom/models'
drive_data_path = '/content/drive/MyDrive/demucs_custom/data'
Path(drive_models_path).mkdir(parents=True, exist_ok=True)
Path(drive_data_path).mkdir(parents=True, exist_ok=True)
print(f"✅ Папка для моделей: {drive_models_path}")
print(f"✅ Папка для данных: {drive_data_path}")

# Датасет держим на ЛОКАЛЬНОМ диске Colab, а не на Drive.
# Причина: 10 ГБ с Drive читаются во время обучения очень медленно
# (это главная причина, почему обучение в Colab встаёт колом).
# На Drive уходят только модель и чекпоинты — их мало, и они должны пережить
# перезапуск сессии.
RAW_MUSDB = "/content/musdb18"
OUT_ROOT = "/content/custom_dataset"
print(f"✅ MUSDB18 качаем сюда:  {RAW_MUSDB}")
print(f"✅ Датасет готовим сюда: {OUT_ROOT}")

In [ ]:
# Скачивание датасета MUSDB18 (~4 ГБ, займёт 5-10 минут)
#
# ВАЖНО: musdb отдаёт треки в своём виде (train/<имя>/vocals.wav и т.д.,
# без mixture.wav). Наш data_prep.py ждёт другую структуру:
#     custom_dataset/train/track_001/{mixture,vocals,drums,bass,other}.wav
#     custom_dataset/valid/track_101/...
# Поэтому ниже скачивание + конвертация в нужный формат.
import glob
import time

import numpy as np
import soundfile as sf

SOURCES = ["vocals", "drums", "bass", "other"]
TARGET_SR = 44100
VALID_RATIO = 0.1        # 10% треков уйдёт в валидацию
MAX_SECONDS = 120        # сколько секунд оставить из каждого трека

# --- Шаг 1: скачивание -----------------------------------------------------
if os.path.isdir(os.path.join(RAW_MUSDB, "train")):
    print("✅ MUSDB18 уже скачан, пропускаем загрузку")
else:
    print("⏳ Скачивание датасета MUSDB18...")
    print("Это займёт 5-10 минут. Можешь пойти попить кофе ☕")
    import musdb
    musdb.DB(root=RAW_MUSDB, download=True)
    print("✅ MUSDB18 скачан!")

raw_tracks = sorted(d for d in glob.glob(os.path.join(RAW_MUSDB, "train", "*")) if os.path.isdir(d))
print(f"Треков в MUSDB18 train: {len(raw_tracks)}")

# --- Шаг 2: конвертация в формат проекта ----------------------------------
def _find_stem(track_dir, name):
    for candidate in (f"{name}.wav", f"{name}.flac", f"{name}.stem.wav"):
        path = os.path.join(track_dir, candidate)
        if os.path.isfile(path):
            return path
    return None

def _read_stereo(path, sr_target=TARGET_SR):
    """Читает аудио и приводит к (каналы, N) float32 нужной частоты."""
    audio, sr = sf.read(path, always_2d=True, dtype="float32")
    if audio.shape[1] > 2:
        audio = audio[:, :2]
    if sr != sr_target:
        import librosa
        audio = librosa.resample(audio.T, orig_sr=sr, target_sr=sr_target).T
    return np.ascontiguousarray(audio.T)

def _write_wav(path, audio):
    """Пишет 16-битный PCM — вдвое меньше места, чем float32."""
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 1.0:
        audio = audio / peak
    sf.write(path, audio.T, TARGET_SR, subtype="PCM_16")

if os.path.isdir(os.path.join(OUT_ROOT, "train")) and os.path.isdir(os.path.join(OUT_ROOT, "valid")):
    print("✅ Датасет уже сконвертирован, пропускаем")
else:
    os.makedirs(os.path.join(OUT_ROOT, "train"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "valid"), exist_ok=True)
    max_samples = int(MAX_SECONDS * TARGET_SR)

    # Фаза 1: отбираем ТОЛЬКО годные треки.
    # Важно: делить на train/valid до фильтрации нельзя — часть треков
    # отбрасывается, и валидация может остаться пустой, а она нужна
    # для выбора лучшей модели.
    usable = []
    for track_dir in raw_tracks:
        stems = {}
        for name in SOURCES:
            path = _find_stem(track_dir, name)
            if path is None:
                break
            stems[name] = path
        if len(stems) != len(SOURCES):
            missing = [n for n in SOURCES if n not in stems]
            print(f"  ⚠️  пропускаю {os.path.basename(track_dir)}: нет {missing}")
            continue
        try:
            frames = min(sf.info(stems[n]).frames for n in SOURCES)
        except Exception as exc:
            print(f"  ⚠️  не читается {os.path.basename(track_dir)}: {type(exc).__name__}")
            continue
        if frames < 10 * TARGET_SR:
            print(f"  ⚠️  пропускаю {os.path.basename(track_dir)}: слишком короткий")
            continue
        usable.append(stems)

    if not usable:
        raise RuntimeError("Ни один трек не прошёл проверку — проверь папку MUSDB18")

    # Фаза 2: режем уже отобранные треки на train и valid.
    n_valid = max(1, int(len(usable) * VALID_RATIO))
    print(f"\nГодных треков: {len(usable)} → train {len(usable) - n_valid}, valid {n_valid}")

    # Фаза 3: конвертируем.
    started = time.time()
    for index, stems in enumerate(usable):
        try:
            audio = {name: _read_stereo(stems[name]) for name in SOURCES}
        except Exception as exc:
            print(f"  ⚠️  не читается трек #{index + 1}: {type(exc).__name__}")
            continue

        length = min(min(a.shape[1] for a in audio.values()), max_samples)
        if length < 10 * TARGET_SR:
            print(f"  ⚠️  пропускаю трек #{index + 1}: слишком короткий")
            continue

        is_valid = index >= len(usable) - n_valid
        split = "valid" if is_valid else "train"
        out_dir = os.path.join(OUT_ROOT, split, f"track_{index + 1:03d}")
        os.makedirs(out_dir, exist_ok=True)

        mixture = np.zeros((2, length), dtype="float32")
        for name in SOURCES:
            chunk = audio[name][:, :length]
            _write_wav(os.path.join(out_dir, f"{name}.wav"), chunk)
            mixture += chunk
        _write_wav(os.path.join(out_dir, "mixture.wav"), mixture)

        done = index + 1
        if done % 10 == 0:
            elapsed = time.time() - started
            print(f"  ... готово {done}/{len(usable)} треков ({elapsed:.0f} сек)")

    print(f"\n✅ Конвертация завершена: {len(usable)} треков")

# --- Шаг 3: сводка --------------------------------------------------------
counts = {}
for split in ("train", "valid"):
    path = os.path.join(OUT_ROOT, split)
    tracks = sorted(d for d in glob.glob(os.path.join(path, "track_*")) if os.path.isdir(d))
    counts[split] = len(tracks)
    print(f"{split}: {len(tracks)} треков")
    if tracks:
        print(f"  пример: {sorted(os.listdir(tracks[0]))}")

if counts["train"] == 0 or counts["valid"] == 0:
    raise RuntimeError(
        f"Датасет неполный: train={counts['train']}, valid={counts['valid']}.\n"
        "Обучение не запустится — train_custom_model нужен и train, и valid."
    )

In [ ]:
# Запуск обучения
from config import Config
from train_custom import train_custom_model

# Настройки для Colab
config = Config()
config.device = "cuda"
config.custom_data_path = OUT_ROOT
config.output_model_path = f'{drive_models_path}/custom_demucs.pth'
config.output_dir = "/content/output"
config.batch_size = 4           # меньше из-за 16 ГБ VRAM в Colab
config.epochs = 30              # начнём с 30 эпох
config.num_workers = 2
config.checkpoint_interval = 10  # ровно как CHECKPOINT_EVERY в train_custom.py
config.early_stopping_patience = 10

problems = config.validate()
if problems:
    for item in problems:
        print("❌", item)
    raise ValueError("Конфиг не прошёл проверку — исправь и перезапусти ячейку")

print("=" * 50)
print("🚀 ЗАПУСК ОБУЧЕНИЯ")
print("=" * 50)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Эпох: {config.epochs}")
print(f"Batch size: {config.batch_size}")
print(f"Датасет: {config.custom_data_path}")
print(f"Модель сохранится в: {config.output_model_path}")
print(f"Чекпоинт: {config.output_model_path_abs.with_name(config.output_model_path_abs.stem + '_last.pth')}")
print("=" * 50)

# assume_yes=True — критично для Colab.
# train_custom_model() обычно зовёт input() с вопросом «продолжить с чекпоинта?».
# В Colab нет stdin → EOFError → обучение молча начинается с нуля.
# С assume_yes=True resume работает автоматически после перезапуска сессии.
best_model_path = train_custom_model(config, assume_yes=True)

print(f"\n🎉 ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print(f"Лучшая модель: {best_model_path}")

# 🎉 Готово!

## Что произошло:
- Модель обучена на GPU Google Colab
- Лучшая версия сохранена в Google Drive: `MyDrive/demucs_custom/models/custom_demucs.pth`
- Чекпоинты тоже там (можно продолжить обучение, если сессия прервётся)

## Что дальше:
1. Скачай `custom_demucs.pth` из Google Drive на свой ПК
2. Положи в папку `demucs_custom/models/` локально
3. Мы протестируем инференс (разделение новой песни)

## Если Colab отключился:
- Не переживай! Чекпоинт сохранён в Google Drive
- Запусти notebook снова, он продолжит с последней эпохи
- Датасет лежит на локальном диске Colab, поэтому качается заново (~5-10 мин),
  но модель и чекпоинт подхватятся из Drive автоматически

## Важно про время обучения:
- Colab даёт GPU T4 примерно на 12 часов, потом сессия умирает
- На T4 30 эпох на 100 треках — это ориентировочно 6-10 часов
- Если не успеешь — просто перезапусти ноутбук, обучение продолжится